## Задача 1

Лесник решил провести кластеризацию животных по их расположению в лесу. Он разделил карту на квадраты по километровым отметкам: первый квадрат можно описать 0 ≤ x ≤ 1, 0 ≤ y ≤ 1, второй — 1 ≤ x ≤ 2,0 ≤ y ≤ 1 и так далее.

Для файла А леснику нужно определить два соседних квадрата, в которых суммарно находится больше всего животных. Для файла Б леснику нужно определить три соседних квадрата. Квадраты называются соседними, если у них есть общая граница.

Для каждого файла вычислите два числа: S — количество социальных животных в выбранных соседних квадратах, и K — количество остальных животных в выбранных соседних квадратах. Животное называется социальным, если в радиусе 0.1 вокруг него находится как минимум 14 других животных.

В ответе запишите четыре числа: в первой строке S и K для файла А, во второй строке аналогичные данные для файла Б.

In [25]:
import numpy as np
from collections import defaultdict

def solve_task1():
    def get_data(filename):
        with open(filename, 'r') as f:
            return [list(map(float, line.split())) for line in f if line.strip()]

    def find_social(points):
        pts = np.array(points)
        social = [False] * len(pts)
        for i in range(len(pts)):
            dist = np.sum((pts - pts[i])**2, axis=1)
            if np.sum(dist <= 0.1**2) >= 15: # 14 других + 1 сама
                social[i] = True
        return social

    def solve_file(data, k_count):
        points = data
        is_social = find_social(points)
        squares = defaultdict(list)
        for i, (x, y) in enumerate(points):
            squares[(int(x), int(y))].append(i)
        
        sq_list = list(squares.keys())
        best_s, best_k = -1, -1

        def are_adj(s1, s2):
            return abs(s1[0]-s2[0]) + abs(s1[1]-s2[1]) == 1

        if k_count == 2:
            for i in range(len(sq_list)):
                for j in range(i+1, len(sq_list)):
                    if are_adj(sq_list[i], sq_list[j]):
                        idx = squares[sq_list[i]] + squares[sq_list[j]]
                        s = sum(is_social[t] for t in idx)
                        k = len(idx) - s
                        if s > best_s or (s == best_s and k > best_k):
                            best_s, best_k = s, k
        else: # k_count == 3
            for i in range(len(sq_list)):
                for j in range(i+1, len(sq_list)):
                    if not are_adj(sq_list[i], sq_list[j]): continue
                    for m in range(j+1, len(sq_list)):
                        if are_adj(sq_list[m], sq_list[i]) or are_adj(sq_list[m], sq_list[j]):
                            idx = squares[sq_list[i]] + squares[sq_list[j]] + squares[sq_list[m]]
                            s = sum(is_social[t] for t in idx)
                            k = len(idx) - s
                            if s > best_s or (s == best_s and k > best_k):
                                best_s, best_k = s, k
        return best_s, best_k

    res_a = solve_file(get_data('1_A.txt'), 2)
    res_b = solve_file(get_data('1_B.txt'), 3)
    return f"{res_a[0]} {res_a[1]}\n{res_b[0]} {res_b[1]}"

In [26]:
print(solve_task1())

104 453
2156 158


## Задача 2

Научно﻿-﻿исследовательский институт проводит мониторинг экологического состояния различных регионов. Результаты измерений представляются в виде пары чисел: первое — концентрация загрязняющего вещества в почве, второе — концентрация того же вещества в близлежащем водоёме. Для анализа результатов эта пара рассматривается как координаты точки на плоскости, и строится график с точками, которые соответствуют всем измерениям.

По ошибке данные нескольких исследуемых регионов были записаны в один файл. Известно, что измерения относятся к одному региону, если они образуют компактные группы точек на графике. Каждая группа лежит внутри прямоугольника высотой H и шириной.

Перед проведением основного анализа необходимо очистить данные от случайных выбросов при измерении. Для этого используется метод межквартильного размаха:
* для каждой группы точек вычисляется первый квартиль Q1 (значение, ниже которого находится 25% измерений) и третий квартиль Q3 (значение, выше которого находится 25% измерений) отдельно для рядов значений концентрации загрязняющего вещества в почве и в близлежащем водоёме (координаты X и Y)
* вычисляется межквартильный размах IQR - Q3 - Q1 для каждой кординаты
* точка считается выбросом, если хотя бы одна из её координат Х или Y выходит за пределы диапазона |Q1 - 1.5 * IQR; Q3 + 1.5 * IQR|

Для каждого региона необходимо рассчитать индекс экологической опасности I, который определяется как отношение среднего значения измерений к их количеству после удаления выбросов.

Под средним значением измерений в этом случае понимается среднее евклидово расстояние между всеми парами различных точек (измерений) в регионе.

В файле A хранятся данные о трёх регионах, где H=30, W=35. Каждая строка файла содержит два числа: координаты X (концентрация в почве) и Y (концентрация в водоёме), соответствующие одному измерению. Значения даны в условных единицах. Известно, что количество измерений не превышает 1000.

В файле B хранятся данные о пяти регионах, где H=40, W=32. Известно, что количество измерений не превышает 10000. Структура хранения информации в файле B аналогична файлу А.

Для каждого файла определите:
* общее количество выявленных выбросов
* регион с максимальным индексом экологической опасности

В ответе запишите четыре числа: в первой строке — общее количество выбросов и целую часть произведения I×100000 для файла A, во второй строке — аналогичные данные для файла B.

Расчёт квартилей
* Найдите медиану значений данных. Это второй квартиль Q2
* Найдите медиану значений данных, которые находятся ниже второго квартиля. Это первый квартиль Q1
* Найдите медиану значений данных, которые выше второго квартиля. Это третий квартиль Q3

In [27]:
def solve_task2():
    def get_quartiles(arr):
        s = sorted(arr)
        def median(data):
            n = len(data)
            if n % 2 == 0: return (data[n//2-1] + data[n//2]) / 2
            return data[n//2]
        q2 = median(s)
        q1 = median([x for x in s if x < q2])
        q3 = median([x for x in s if x > q2])
        return q1, q3

    def process(filename, H, W):
        with open(filename, 'r') as f:
            pts = [list(map(float, l.split())) for l in f if l.strip()]

        used = [False] * len(pts)
        total_outliers = 0
        max_I = 0
        for i in range(len(pts)):
            if used[i]: continue
            cluster = [pts[i]]
            used[i] = True
            for j in range(len(pts)):
                if not used[j] and abs(pts[i][0]-pts[j][0]) <= W and abs(pts[i][1]-pts[j][1]) <= H:
                    cluster.append(pts[j]); used[j] = True
            
            if len(cluster) < 2: continue
            
            xs, ys = [p[0] for p in cluster], [p[1] for p in cluster]
            q1x, q3x = get_quartiles(xs)
            q1y, q3y = get_quartiles(ys)
            iqr_x, iqr_y = q3x - q1x, q3y - q1y
            
            good = []
            out_cnt = 0
            for p in cluster:
                if (q1x - 1.5*iqr_x <= p[0] <= q3x + 1.5*iqr_x) and \
                   (q1y - 1.5*iqr_y <= p[1] <= q3y + 1.5*iqr_y):
                    good.append(p)
                else: out_cnt += 1
            
            total_outliers += out_cnt
            if len(good) > 1:
                dists = []
                for k in range(len(good)):
                    for m in range(k+1, len(good)):
                        dists.append(((good[k][0]-good[m][0])**2 + (good[k][1]-good[m][1])**2)**0.5)
                I = np.mean(dists) / len(good)
                max_I = max(max_I, I)
        return total_outliers, int(max_I * 100000)

    a = process('2_A.txt', 30, 35)
    b = process('2_B.txt', 40, 32)
    return f"{a[0]} {a[1]}\n{b[0]} {b[1]}"

In [28]:
print(solve_task2())

25 3804
467 189


## Задача 3

Кластеризуйте как в предыдущих домашках. Будем называть центром кластера точку этого кластера, сумма расстояний от которой до всех остальных точек кластера минимальна.

Для каждого файла определите координаты центра каждого кластера, затем вычислите два числа: A - среднее арифметическое абсцисс центров кластеров, и среднее B - арифметическое ординат центров кластеров. В ответе запишите четыре числа: в первой строке сначала целую часть произведения A\*10000 затем целую часть произведения B\*10000 для файла А, во второй строке — аналогичные данные для файла Б.


In [29]:
def solve_task3():
    def find_centers(filename):
        with open(filename, 'r') as f:
            data = np.array([list(map(float, l.replace(',','.').split())) for l in f if l.strip()])
        
        from sklearn.cluster import dbscan
        _, labels = dbscan(data, eps=5, min_samples=1)
        
        centers = []
        for lbl in set(labels):
            cluster = data[labels == lbl]
            sums = [np.sum(np.linalg.norm(cluster - p, axis=1)) for p in cluster]
            centers.append(cluster[np.argmin(sums)])
        
        c = np.array(centers)
        return int(np.mean(c[:,0])*10000), int(np.mean(c[:,1])*10000)

    a = find_centers('3_A.txt')
    b = find_centers('3_B.txt')
    return f"{a[0]} {a[1]}\n{b[0]} {b[1]}"


In [30]:
print(solve_task3())

254624 48396
-52968 63812


## Задача 4

Исследователь анализирует набор объектов, каждый из которых характеризуется пятью числовыми параметрами. Он знает, что объекты образуют несколько групп (кластеров), которые можно выявить при проекции на плоскость только двух параметров из пяти. Значения в одном из столбцов будут соответствовать координатам по оси абсцисс, а из второго — координатам по оси ординат. Каждый кластер можно заключить в квадратную область заданного размера L, причём эти квадраты между собой не пересекаются. Стороны квадратов параллельны координатным осям. Каждый объект должен принадлежать только одному кластеру.

Евклидово расстояние

В файле A хранятся данные о наборе объектов, образующих три кластера. В каждой строке через пробел записаны пять параметров, характеризующих один объект. Все значения представлены с точностью до двух знаков после запятой. Количество объектов в файле А не превышает 1000.

В файле Б записаны данные о наборе объектов, образующих шесть кластеров, с аналогичной структурой хранения информации. Количество объектов в файле Б не превышает 10000.

Для каждого файла необходимо определить, какая пара параметров позволяет разделить объекты на кластеры, и найти минимальный размер стороны квадрата L, который может содержать все точки одного кластера при проекции на плоскость найденных параметров. Также определите в каждом кластере расстояние между двумя объектами, расположенными дальше всего друг от друга, и вычислите P — среднее арифметическое таких расстояний для всех кластеров.

В ответе запишите четыре числа: в первой строке — целую часть произведения L×10000, затем целую часть произведения P×10000 для файла А, во второй строке — аналогичные значения для файла Б.

In [31]:
def solve_task4():
    def get_data(filename):
        try:
            with open(filename, 'r') as f:
                return np.array([list(map(float, l.replace(',', '.').split())) for l in f if l.strip()])
        except FileNotFoundError:
            return None

    def get_clusters(points, L):
        n = len(points)
        labels = -np.ones(n)
        cluster_id = 0
        for i in range(n):
            if labels[i] == -1:
                labels[i] = cluster_id
                stack = [i]
                while stack:
                    curr = stack.pop()
                    neighbors = np.where((labels == -1) & 
                                         (np.abs(points[:,0] - points[curr,0]) <= L) & 
                                         (np.abs(points[:,1] - points[curr,1]) <= L))[0]
                    for neighbor in neighbors:
                        labels[neighbor] = cluster_id
                        stack.append(neighbor)
                cluster_id += 1
        return labels, cluster_id

    def process_file(filename, target_k):
        data = get_data(filename)
        if data is None: return 0, 0
        
        best_L, best_P = 0.0, 0.0
        min_P = float('inf')

        for p1, p2 in combinations(range(5), 2):
            proj = data[:, [p1, p2]]

            all_dists = sorted(np.unique(np.abs(proj[0,0] - proj[:,0])))
            
            for L in np.linspace(0.1, 10.0, 50): 
                labels, k_found = get_clusters(proj, L)
                
                if k_found == target_k:
                    current_P = 0
                    for c_id in range(target_k):
                        cluster_pts = proj[labels == c_id]
                        if len(cluster_pts) > 1:
                            center = np.mean(cluster_pts, axis=0)
                            current_P += np.mean(np.linalg.norm(cluster_pts - center, axis=1))
                    
                    avg_P = current_P / target_k
                    
                    if avg_P < min_P:
                        min_P = avg_P
                        best_L = L
                        best_P = avg_P
                        
        return best_L, best_P

    L_a, P_a = process_file('4_A.txt', target_k=3)
    L_b, P_b = process_file('4_B.txt', target_k=5)

    return f"{int(L_a * 10000)} {int(P_a * 10000)}\n{int(L_b * 10000)} {int(P_b * 10000)}"

In [32]:
print(solve_task4())

27265 12155
69693 24788


In [33]:
## Задача 5

In [34]:
def solve_task5():
    def load_data(filename):
        try:
            return np.loadtxt(filename)
        except FileNotFoundError:
            print(f"Ошибка: Файл {filename} не найден.")
            return None
        except Exception as e:
            print(f"Ошибка при чтении {filename}: {e}")
            return None

    def get_clusters(points, L):
        n = len(points)
        labels = -np.ones(n)
        cluster_id = 0
        for i in range(n):
            if labels[i] == -1:
                labels[i] = cluster_id
                stack = [i]
                while stack:
                    curr = stack.pop()
                    neighbors = np.where((labels == -1) & 
                                         (np.abs(points[:,0] - points[curr,0]) <= L) & 
                                         (np.abs(points[:,1] - points[curr,1]) <= L))[0]
                    for neighbor in neighbors:
                        labels[neighbor] = cluster_id
                        stack.append(neighbor)
                cluster_id += 1
        return labels, cluster_id

    def process_file(filename, target_k):
        data = load_data(filename)
        if data is None: return 0.0, 0.0

        num_cols = data.shape[1]
        best_L, best_P = 0.0, 0.0
        min_P = float('inf')

        for p1, p2 in combinations(range(num_cols), 2):
            proj = data[:, [p1, p2]]

            for L in np.arange(0.01, 10.0, 0.05):
                labels, k_found = get_clusters(proj, L)
                
                if k_found == target_k:
                    p_sum = 0
                    for c_id in range(target_k):
                        cluster_pts = proj[labels == c_id]
                        if len(cluster_pts) > 0:
                            center = np.mean(cluster_pts, axis=0)
                            p_sum += np.mean(np.linalg.norm(cluster_pts - center, axis=1))
                    
                    avg_P = p_sum / target_k
                    if avg_P < min_P or min_P == float('inf'):
                        min_P = avg_P
                        best_L, best_P = L, avg_P
                    break
                    
        return best_L, best_P
    
    L_a, P_a = process_file('5_A.txt', target_k=3)
    L_b, P_b = process_file('5_B.txt', target_k=5)

    return f"{int(L_a * 10000)} {int(P_a * 10000)}\n{int(L_b * 10000)} {int(P_b * 10000)}"


In [35]:
print(solve_task5())

3600 2500
3600 2484
